In [1]:
import os
import matplotlib as mpl
import numpy as np
import pandas as pd
import scanpy as sc
import celltypist
from celltypist import models


sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=80, facecolor='white', color_map='viridis')
sc.logging.print_header()

scanpy==1.9.1 anndata==0.8.0 umap==0.5.3 numpy==1.20.1 scipy==1.6.1 pandas==1.4.3 scikit-learn==0.24.1 statsmodels==0.13.2 python-igraph==0.8.3 louvain==0.7.0 leidenalg==0.8.3 pynndescent==0.5.2


In [25]:
adata = sc.read_h5ad('/nfs/team205/bh14/Datasets/Remapped/raw_adata/QC_concatenated/Immune_compartment_Concatenated_dataset.h5ad')


In [26]:
adata.obs['dataset_id'].value_counts()

GSE157278_Hong_2020            60205
GSE163314_Lefferts_2021        53037
E-MTAB-9492_Penkava_2020       33698
GSE134809_Martin_et_al_2019    26855
Braga_asthma_CD4                 929
Name: dataset_id, dtype: int64

In [27]:
adata.shape

(174724, 36601)

In [28]:
sc.pp.filter_cells(adata, min_counts=1000)

filtered out 12312 cells that have less than 1000 counts


In [29]:
sc.pp.filter_cells(adata, min_genes=600)

filtered out 5490 cells that have less than 600 genes expressed


In [30]:
adata = adata[adata.obs['QC'] == 'Pass']

In [31]:
adata.shape

(152453, 36601)

In [32]:
adata.obs['dataset_id'].value_counts()

GSE157278_Hong_2020            53206
GSE163314_Lefferts_2021        46985
E-MTAB-9492_Penkava_2020       32821
GSE134809_Martin_et_al_2019    18524
Braga_asthma_CD4                 917
Name: dataset_id, dtype: int64

In [33]:
# Remove donors with few than 50 cells
donor_counts = adata.obs['donor_id'].value_counts()

In [34]:
donor_counts.sort_values()[:30]

GSM4761136                        4
ARMS014                          17
ARMS035                          25
ARMS038                          34
ARMS033                          44
ARMS024                          71
ARMS009                          98
ARMS018                         101
ARMS015                         114
ARMS004                         200
ARMS005                         213
GSM4761138                      879
GSM4761139                     1067
GSM4761137                     1920
GSM4761140                     2848
HC-5_GSM4760624                3055
GSM4761142                     3450
Patient_21_Blood_GSM4977001    3598
GSM4761141                     3744
Patient_33_Blood_GSM4977007    3776
pSS-4_GSM4760628               4243
pSS-2_GSM4760626               4333
pSS-3_GSM4760627               4350
HC-1_GSM4760620                4448
Patient_2_Blood_GSM4976993     4594
GSM4761144                     4612
Patient_3_Blood_GSM4976995     5362
HC-2_GSM4760621             

In [35]:
donor_counts[donor_counts<50].sum()

124

In [36]:
adata = adata[adata.obs['donor_id'].isin( donor_counts[donor_counts>=50].index)]

In [37]:
adata.layers["counts"] = adata.X.copy()

In [38]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

normalizing counts per cell
    finished (0:00:00)


In [39]:
adata.raw = adata  # keep full dimension safe

In [40]:
adata.X.expm1().sum(axis = 1)

matrix([[10000.   ],
        [10000.001],
        [10000.   ],
        ...,
        [10000.   ],
        [10000.   ],
        [10000.   ]], dtype=float32)

#### Celltypist

In [18]:
predictions = celltypist.annotate(adata, model = 'Immune_All_Low.pkl', majority_voting = True)

🔬 Input data has 138539 cells and 16855 genes
🔗 Matching reference genes in the model
🧬 5240 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering


If you pass `n_top_genes`, all cutoffs are ignored.
extracting highly variable genes
    finished (0:00:06)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)
... as `zero_center=True`, sparse input is densified and may lead to large memory consumption
computing PCA
    on highly variable genes
    with n_comps=50
    finished (0:00:34)
computing neighbors
    using 'X_pca' with n_pcs = 50
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:26)


⛓️ Over-clustering input data with resolution set to 25


running Leiden clustering
    finished: found 387 clusters and added
    'over_clustering', the cluster labels (adata.obs, categorical) (0:01:31)


🗳️ Majority voting the predictions
✅ Majority voting done!


In [19]:
predictions.predicted_labels

,predicted_labels,over_clustering,majority_voting
AAACCTGAGCTTCGCG-0_0_0_0,CD16- NK cells,280,CD16- NK cells
AAACCTGAGTTGCAGG-0_0_0_0,Plasmablasts,316,Proliferative germinal center B cells
AAACCTGCAGATCTGT-0_0_0_0,Naive B cells,117,Naive B cells
AAACCTGCATCCGGGT-0_0_0_0,Memory B cells,2,Naive B cells
AAACCTGCATTGTGCA-0_0_0_0,Naive B cells,2,Naive B cells
...,...,...,...
TTTGTCATCGGCTTGG-1-1-1-1_1,Tem/Temra cytotoxic T cells,30,Tem/Trm cytotoxic T cells
TTTGTCATCGTACGGC-1-1-1-1_1,Regulatory T cells,191,Tem/Trm cytotoxic T cells
TTTGTCATCGTTACAG-1-1-1-1_1,Tem/Effector helper T cells,71,Tem/Effector helper T cells
TTTGTCATCTGCGTAA-1-1-1-1_1,Tem/Effector helper T cells,42,Tem/Effector helper T cells


In [20]:
# Get an `AnnData` with predicted labels embedded into the cell metadata columns.
adata_celltypist = predictions.to_adata()

In [21]:
adata_celltypist

AnnData object with n_obs × n_vars = 138539 × 16855
    obs: 'dataset_id', 'donor_id', 'tissue', 'cell_source', 'disease', 'doublet_scores', 'predicted_doublets', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_rb', 'pct_counts_rb', 'QC', 'barcodes', 'disease_status', 'age', 'sex', 'developmental_stage', 'batch', 'development_stage', 'n_counts', 'n_genes', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score'
    var: 'mt', 'rb', 'n_cells-0-0-0', 'n_cells_by_counts-0-0-0', 'mean_counts-0-0-0', 'pct_dropout_by_counts-0-0-0', 'total_counts-0-0-0', 'n_cells-1-0-0', 'n_cells_by_counts-1-0-0', 'mean_counts-1-0-0', 'pct_dropout_by_counts-1-0-0', 'total_counts-1-0-0', 'n_cells-1-0', 'n_cells_by_counts-1-0', 'mean_counts-1-0', 'pct_dropout_by_counts-1-0', 'total_counts-1-0', 'n_cells-1', 'n_cells_by_counts-1', 'mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1'
    uns: 'log1p', 'neighbors', 'leiden'
    obsm: 'X_pca'
    layers:

In [22]:
adata_celltypist.obs['predicted_labels'].value_counts()

Tem/Trm cytotoxic T cells           19001
Alveolar macrophages                16770
Tem/Effector helper T cells         15294
Naive B cells                        9019
Regulatory T cells                   7724
                                    ...  
Tem/Effector helper T cells PD1+        1
Cycling DCs                             1
MEMP                                    1
Megakaryocyte precursor                 1
Megakaryocytes/platelets                1
Name: predicted_labels, Length: 87, dtype: int64

In [23]:
adata_celltypist.write_h5ad('/nfs/team205/bh14/Datasets/Remapped/raw_adata/Celltypist/Immune_compartment_sorted_from_Tissue_celltypist.h5ad')
